# End-to-End Cryptanalysis Demonstration
This notebook demonstrates the full pipeline of our automated cryptanalysis system:
1. **Input Ciphertext**: Taking an unknown encrypted text.
2. **Identify Cipher**: Extracting features and predicting the cipher type using our trained machine learning model.
3. **Break Cipher**: Routing to the appropriate algorithmic breaker to recover the key and plaintext.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import time
import pandas as pd
import matplotlib.pyplot as plt

# Import custom modules
from src.ciphers.caesar import CaesarCipher
from src.ciphers.vigenere import VigenereCipher
from src.ciphers.substitution import SubstitutionCipher
from src.ciphers.affine import AffineCipher
from src.ciphers.columnar_transposition import ColumnarTranspositionCipher
from src.ciphers.playfair import PlayfairCipher

from src.breakers.caesar_breaker import CaesarBreaker
from src.breakers.vigenere_breaker import VigenereBreaker
from src.breakers.substitution_breaker import SubstitutionBreaker
from src.breakers.affine_breaker import AffineBreaker
from src.breakers.columnar_transposition_breaker import ColumnarTranspositionBreaker
from src.breakers.playfair_breaker import PlayfairBreaker

from src.ml.feature_extraction import FeatureExtractor
from src.ml.model import CipherClassifier
from src.language.quadgram_scorer import QuadgramScorer

plt.style.use('seaborn-v0_8-darkgrid')

## Setup: Feature Extractor and Cipher Classifier

In [ ]:
# Initialize scorer and extractor
print("Initializing components...")
scorer = QuadgramScorer()
extractor = FeatureExtractor()
classifier = CipherClassifier(model_type='rf')

try:
    classifier.load('models/cipher_classifier_rf.joblib')
    print("Model loaded successfully.")
except:
    print("Model not found. Please train the model using notebook 3 or the data generation scripts.")
    # To make demo work without pretrained, we could mock or provide instructions.
    print("You may need to run data generation and training first.")

## Helper Function: analyze_and_break
This function brings together the prediction and the breaking stages.

In [ ]:
def analyze_and_break(ciphertext):
    start_time = time.time()
    
    # 1. Feature Extraction
    features = extractor.extract(ciphertext)
    
    # 2. Prediction
    pred, conf = classifier.predict(features)
    
    print(f"Predicted Cipher: {pred} (Confidence: {conf:.2f})")
    
    # 3. Breaking
    recovered_text = ""
    recovered_key = None
    
    if pred == 'caesar':
        breaker = CaesarBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    elif pred == 'vigenere':
        breaker = VigenereBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    elif pred == 'substitution':
        breaker = SubstitutionBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    elif pred == 'affine':
        breaker = AffineBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    elif pred == 'columnar_transposition':
        breaker = ColumnarTranspositionBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    elif pred == 'playfair':
        breaker = PlayfairBreaker(scorer)
        recovered_key, recovered_text = breaker.break_cipher(ciphertext)
    else:
        print("Unknown cipher type.")
        
    end_time = time.time()
    
    return {
        'cipher_type': pred,
        'confidence': conf,
        'plaintext': recovered_text,
        'key': recovered_key,
        'time': end_time - start_time
    }

results_history = []

## Demo 1: Caesar Cipher

In [ ]:
pt_caesar = 'THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG'
ct_caesar = CaesarCipher.encrypt(pt_caesar, 7)
print(f"Ciphertext: {ct_caesar}")

res_caesar = analyze_and_break(ct_caesar)
res_caesar['true_type'] = 'caesar'
res_caesar['decryption_success'] = (res_caesar['plaintext'][:20].lower() == pt_caesar[:20].lower().replace(' ', ''))
results_history.append(res_caesar)

print(f"Recovered Key: {res_caesar['key']}")
print(f"Recovered Plaintext: {res_caesar['plaintext'][:50]}...")

## Demo 2: Vigenère Cipher

In [ ]:
pt_vig = 'THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG MULTIPLE TIMES TO MAKE THE TEXT LONG ENOUGH FOR ANALYSIS'
ct_vig = VigenereCipher.encrypt(pt_vig, 'SECRET')
print(f"Ciphertext: {ct_vig}")

res_vig = analyze_and_break(ct_vig)
res_vig['true_type'] = 'vigenere'
res_vig['decryption_success'] = (res_vig['plaintext'][:20].lower() == pt_vig[:20].lower().replace(' ', ''))
results_history.append(res_vig)

print(f"Recovered Key: {res_vig['key']}")
print(f"Recovered Plaintext: {res_vig['plaintext'][:50]}...")

## Demo 3: Substitution Cipher
Note: Breaking substitution ciphers involves hill climbing and may take a few seconds.

In [ ]:
import random
import string

alphabet = list(string.ascii_uppercase)
shuffled = alphabet.copy()
random.shuffle(shuffled)
key_sub = ''.join(shuffled)

pt_sub = 'THIS IS A LONGER TEXT MEANT TO TEST THE CAPABILITIES OF THE HILL CLIMBING ALGORITHM ON A GENERAL SUBSTITUTION CIPHER'
ct_sub = SubstitutionCipher.encrypt(pt_sub, key_sub)
print(f"Ciphertext: {ct_sub}")

res_sub = analyze_and_break(ct_sub)
res_sub['true_type'] = 'substitution'
res_sub['decryption_success'] = (res_sub['plaintext'][:20].lower() == pt_sub[:20].lower().replace(' ', ''))
results_history.append(res_sub)

print(f"Recovered Key: {res_sub['key']}")
print(f"Recovered Plaintext: {res_sub['plaintext'][:50]}...")

## Demo 4: Affine Cipher

In [ ]:
pt_affine = 'THE AFFINE CIPHER IS A TYPE OF MONOALPHABETIC SUBSTITUTION CIPHER'
ct_affine = AffineCipher.encrypt(pt_affine, (5, 8))
print(f"Ciphertext: {ct_affine}")

res_affine = analyze_and_break(ct_affine)
res_affine['true_type'] = 'affine'
res_affine['decryption_success'] = (res_affine['plaintext'][:20].lower() == pt_affine[:20].lower().replace(' ', ''))
results_history.append(res_affine)

print(f"Recovered Key: {res_affine['key']}")
print(f"Recovered Plaintext: {res_affine['plaintext'][:50]}...")

## Demo 5: Columnar Transposition

In [ ]:
pt_col = 'TRANSPOSITION CIPHERS SHIFT THE POSITIONS OF THE CHARACTERS INSTEAD OF CHANGING THEM'
ct_col = ColumnarTranspositionCipher.encrypt(pt_col, 'ZEBRA')
print(f"Ciphertext: {ct_col}")

res_col = analyze_and_break(ct_col)
res_col['true_type'] = 'columnar_transposition'
res_col['decryption_success'] = (res_col['plaintext'][:20].lower() == pt_col[:20].lower().replace(' ', ''))
results_history.append(res_col)

print(f"Recovered Key: {res_col['key']}")
print(f"Recovered Plaintext: {res_col['plaintext'][:50]}...")

## Demo 6: Playfair Cipher

In [ ]:
pt_playfair = 'THE PLAYFAIR CIPHER ENCRYPTS PAIRS OF LETTERS INSTEAD OF SINGLE LETTERS'
ct_playfair = PlayfairCipher.encrypt(pt_playfair, 'MONARCHY')
print(f"Ciphertext: {ct_playfair}")

res_playfair = analyze_and_break(ct_playfair)
res_playfair['true_type'] = 'playfair'
res_playfair['decryption_success'] = (res_playfair['plaintext'][:20].lower() == pt_playfair[:20].lower().replace(' ', ''))
results_history.append(res_playfair)

print(f"Recovered Key: {res_playfair['key']}")
print(f"Recovered Plaintext: {res_playfair['plaintext'][:50]}...")

## Summary Table

In [ ]:
df_results = pd.DataFrame(results_history)
df_results['correctly_identified'] = df_results['cipher_type'] == df_results['true_type']
display(df_results[['true_type', 'cipher_type', 'correctly_identified', 'confidence', 'decryption_success', 'time']])